# Analyse Exploratoire des Donnees (EDA)
## ACPE Match — IndabaX Congo 2026

**Objectif** : Explorer et comprendre les donnees des demandeurs d'emploi et des offres d'emploi
fournies par l'Agence Congolaise pour l'Emploi (ACPE).

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.size'] = 10

DB_PATH = 'acpe.db'
conn = sqlite3.connect(DB_PATH)
print('Connexion a la base SQLite etablie.')

---
## 1. Vue d'ensemble des donnees

In [ ]:
df_cand = pd.read_sql('SELECT * FROM candidates', conn)
df_off = pd.read_sql('SELECT * FROM job_offers', conn)

print(f'Demandeurs d emploi : {len(df_cand):,} enregistrements, {df_cand.shape[1]} colonnes')
print(f'Offres d emploi    : {len(df_off):,} enregistrements, {df_off.shape[1]} colonnes')
print(f'Ratio candidats/offres : {len(df_cand)/len(df_off):.1f}:1')

print('\n--- Colonnes candidats ---')
print(list(df_cand.columns))

print('\n--- Colonnes offres ---')
print(list(df_off.columns))

In [ ]:
print('=== Echantillon candidats ===')
df_cand[['id', 'genre', 'age', 'etudes', 'metier_vise', 'secteur_demande', 'mobilite']].head(10)

In [ ]:
print('=== Echantillon offres ===')
df_off[['id', 'intitule', 'type_contrat', 'entreprise', 'secteur', 'localisation']].head(10)

---
## 2. Analyse des donnees manquantes

In [ ]:
def missing_table(df, name):
    total = len(df)
    missing = df.isnull().sum() + (df == '').sum()
    pct = (missing / total * 100).round(1)
    result = pd.DataFrame({'Colonnes': df.columns, 'Manquants': missing.values, 'Pourcentage (%)': pct.values})
    result = result[result['Manquants'] > 0].sort_values('Pourcentage (%)', ascending=False)
    if len(result) > 0:
        print(f'\nDonnees manquantes dans {name} ({total:,} lignes) :')
        display(result)
    else:
        print(f'\nAucune donnee manquante dans {name}.')
    return result

missing_cand = missing_table(df_cand, 'Candidats')
missing_off = missing_table(df_off, 'Offres')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (df, title) in zip(axes, [(df_cand, 'Candidats'), (df_off, 'Offres')]):
    total = len(df)
    missing_pct = (df.isnull().sum() + (df == '').sum()) / total * 100
    missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=True)
    if len(missing_pct) > 0:
        missing_pct.plot(kind='barh', ax=ax, color='#e74c3c')
        ax.set_xlabel('% Manquant')
        ax.set_title(f'Donnees manquantes — {title}')
        ax.set_xlim(0, 105)
    else:
        ax.text(0.5, 0.5, 'Aucune donnee manquante', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f'Donnees manquantes — {title}')

plt.tight_layout()
plt.savefig('eda_01_missing_data.png', dpi=100, bbox_inches='tight')
plt.show()
print('Graphique sauvegarde: eda_01_missing_data.png')

**Observations** :
- Candidats : `nom` et `prenom` sont 100% vides (donnees anonymisees). Tous les autres champs sont completes.
- Offres : `description` et `competences_recherchees` sont ~94% vides — defi majeur pour le matching par skills.

---
## 3. Analyse des candidats

### 3.1 Repartition par genre

In [ ]:
genre_counts = df_cand['genre'].value_counts()
fig, ax = plt.subplots(figsize=(6, 5))
genre_counts.plot(kind='pie', autopct='%1.1f%%', colors=['#3498db', '#e91e63', '#95a5a6'], ax=ax)
ax.set_ylabel('')
ax.set_title('Repartition par genre')
plt.tight_layout()
plt.savefig('eda_02_genre.png', dpi=100, bbox_inches='tight')
plt.show()
print(genre_counts)

### 3.2 Distribution de l'age

In [ ]:
age_valid = df_cand['age'][(df_cand['age'] > 0) & (df_cand['age'] < 100)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(age_valid, bins=30, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[0].axvline(age_valid.median(), color='red', linestyle='--', label=f'Mediane: {age_valid.median():.0f} ans')
axes[0].axvline(age_valid.mean(), color='blue', linestyle='--', label=f'Moyenne: {age_valid.mean():.1f} ans')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Nombre de candidats')
axes[0].set_title('Distribution des ages')
axes[0].legend()

age_valid.plot(kind='box', ax=axes[1], vert=True)
axes[1].set_ylabel('Age')
axes[1].set_title('Boxplot des ages')

plt.tight_layout()
plt.savefig('eda_03_age.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Stats age: min={age_valid.min()}, max={age_valid.max()}, mediane={age_valid.median():.0f}, moyenne={age_valid.mean():.1f}')

### 3.3 Niveau d'etudes

In [ ]:
edu_order = ['NV_0_AUCUN', 'NV_1_PRIMARY', 'NV_2_COLLEGE', 'NV_3_PRO_N1',
             'NV_4_BAC', 'NV_5_BAC_2', 'NV_6_BAC_3', 'NV_7_BAC_5', 'NV_8_DOCTORAT']
edu_labels = ['Aucun', 'Primaire', 'College', 'Pro N1', 'Bac', 'Bac+2', 'Bac+3', 'Bac+5', 'Doctorat']

edu_counts = df_cand['code_niveau_etude'].value_counts().reindex(edu_order).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(edu_counts)), edu_counts.values, color='#9b59b6', edgecolor='white')
ax.set_xticks(range(len(edu_counts)))
ax.set_xticklabels(edu_labels, rotation=30, ha='right')
ax.set_ylabel('Nombre de candidats')
ax.set_title('Repartition par niveau d etudes')

for bar, val in zip(bars, edu_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_04_education.png', dpi=100, bbox_inches='tight')
plt.show()
print(edu_counts)

### 3.4 Metiers les plus demandes

In [ ]:
metier_counts = df_cand['metier_vise'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
metier_counts.plot(kind='barh', color='#e67e22', edgecolor='white', ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Nombre de candidats')
ax.set_title('Top 15 des metiers vises par les candidats')

for i, v in enumerate(metier_counts.values):
    ax.text(v + 20, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_05_metiers_candidats.png', dpi=100, bbox_inches='tight')
plt.show()

### 3.5 Secteur d'activite demande

In [ ]:
sec_counts = df_cand['secteur_demande'].value_counts().head(12)

fig, ax = plt.subplots(figsize=(12, 6))
sec_counts.plot(kind='barh', color='#1abc9c', edgecolor='white', ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Nombre de candidats')
ax.set_title('Top 12 des secteurs demandes par les candidats')

for i, v in enumerate(sec_counts.values):
    ax.text(v + 50, i, f'{v:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_06_secteurs_candidats.png', dpi=100, bbox_inches='tight')
plt.show()

### 3.6 Mobilite geographique

In [ ]:
mob_counts = df_cand['mobilite'].value_counts()
print('Mobilite geographique :')
print(mob_counts)
print(f'\nNote : 90.6% des candidats ont la mobilite non renseignee.')
print(f'Parmi ceux qui ont repondu : {mob_counts.get("Non", 0):,} non / {mob_counts.get("Oui", 0):,} oui')

---
## 4. Analyse des offres

### 4.1 Types de contrat

In [ ]:
contrat_counts = df_off['type_contrat'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
contrat_counts.plot(kind='bar', color='#3498db', edgecolor='white', ax=ax)
ax.set_ylabel('Nombre d offres')
ax.set_title('Repartition par type de contrat')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')

for i, v in enumerate(contrat_counts.values):
    ax.text(i, v + 10, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_07_contrats.png', dpi=100, bbox_inches='tight')
plt.show()
print(contrat_counts)

### 4.2 Secteurs d'activite

In [ ]:
sect_counts = df_off['secteur'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
sect_counts.plot(kind='barh', color='#e74c3c', edgecolor='white', ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Nombre d offres')
ax.set_title('Top 15 des secteurs d activite (offres)')

for i, v in enumerate(sect_counts.values):
    ax.text(v + 3, i, f'{v}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_08_secteurs_offres.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.3 Repartition geographique des offres

In [ ]:
loc_counts = df_off['localisation'].value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

loc_counts.plot(kind='barh', color='#2c3e50', edgecolor='white', ax=axes[0])
axes[0].invert_yaxis()
axes[0].set_xlabel('Nombre d offres')
axes[0].set_title('Top 15 localisations des offres')

loc_pct = loc_counts.head(3)
axes[1].pie(loc_pct.values, labels=loc_pct.index, autopct='%1.1f%%',
            colors=['#2c3e50', '#7f8c8d', '#bdc3c7'])
axes[1].set_title('Top 3 villes (part des offres)')

plt.tight_layout()
plt.savefig('eda_09_localisation_offres.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Total offres : {len(df_off):,}')
print(f'POINTE-NOIRE : {loc_counts.get("POINTE-NOIRE", 0):,} ({loc_counts.get("POINTE-NOIRE", 0)/len(df_off)*100:.1f}%)')
print(f'BRAZZAVILLE  : {loc_counts.get("BRAZZAVILLE", 0):,} ({loc_counts.get("BRAZZAVILLE", 0)/len(df_off)*100:.1f}%)')

### 4.4 Entreprises avec le plus d'offres

In [ ]:
ent_counts = df_off['entreprise'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
ent_counts.plot(kind='barh', color='#8e44ad', edgecolor='white', ax=ax)
ax.invert_yaxis()
ax.set_xlabel('Nombre d offres')
ax.set_title('Top 15 entreprises recruteuses')

for i, v in enumerate(ent_counts.values):
    ax.text(v + 1, i, f'{v}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('eda_10_entreprises.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 5. Analyse croisee Candidats vs Offres

### 5.1 Chevauchement des secteurs

In [ ]:
cand_sectors = set(df_cand['secteur_demande'].unique())
off_sectors = set(df_off['secteur'].unique())

common = cand_sectors & off_sectors
cand_only = cand_sectors - off_sectors
off_only = off_sectors - cand_sectors

print(f'Secteurs demandes par les candidats : {len(cand_sectors)}')
print(f'Secteurs des offres                  : {len(off_sectors)}')
print(f'Secteurs en commun                   : {len(common)}')
print(f'Secteurs candidats uniquement        : {len(cand_only)}')
print(f'Secteurs offres uniquement           : {len(off_only)}')

if cand_only:
    print(f'\nSecteurs sans offre correspondante :')
    for s in sorted(cand_only):
        n = len(df_cand[df_cand['secteur_demande'] == s])
        print(f'  - {s} ({n:,} candidats)')

if off_only:
    print(f'\nOffres sans candidat interesse :')
    for s in sorted(off_only):
        n = len(df_off[df_off['secteur'] == s])
        print(f'  - {s} ({n} offres)')

In [ ]:
cand_sec_top10 = df_cand['secteur_demande'].value_counts().head(10)
off_sec_top10 = df_off['secteur'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

cand_sec_top10.plot(kind='barh', color='#3498db', ax=axes[0])
axes[0].invert_yaxis()
axes[0].set_title('Candidats — Top 10 secteurs demandes')
axes[0].set_xlabel('Nombre de candidats')

off_sec_top10.plot(kind='barh', color='#e74c3c', ax=axes[1])
axes[1].invert_yaxis()
axes[1].set_title('Offres — Top 10 secteurs')
axes[1].set_xlabel('Nombre d offres')

plt.tight_layout()
plt.savefig('eda_11_sector_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

### 5.2 Top metiers : candidats vs offres

In [ ]:
cand_metiers = set(df_cand['metier_vise'].str.lower().str.strip().unique())
off_intitules = set(df_off['intitule'].str.lower().str.strip().unique())

common_metiers = cand_metiers & off_intitules

print(f'Metiers vises par les candidats : {len(cand_metiers)} uniques')
print(f'Intitules d offres              : {len(off_intitules)} uniques')
print(f'Match exact (metier = intitule) : {len(common_metiers)}')

if common_metiers:
    print(f'\nExemples de matchs exacts :')
    for m in sorted(common_metiers)[:20]:
        nc = len(df_cand[df_cand['metier_vise'].str.lower().str.strip() == m])
        no = len(df_off[df_off['intitule'].str.lower().str.strip() == m])
        print(f'  {m} : {nc:,} candidats / {no} offres')

---
## 6. Qualite des donnees et observations

In [ ]:
print('=== RESUME DE L ANALYSE EXPLORATOIRE ===')
print()
print(f'DONNEES :')
print(f'  - {len(df_cand):,} candidats (41,285 enregistrements anonymises)')
print(f'  - {len(df_off):,} offres d emploi')
print(f'  - Ratio candidats/offres : {len(df_cand)/len(df_off):.1f}:1')
print()
print(f'CANDIDATS :')
print(f'  - Genre : {genre_counts.get("Homme", 0):,} hommes ({genre_counts.get("Homme", 0)/len(df_cand)*100:.1f}%), {genre_counts.get("Femme", 0):,} femmes ({genre_counts.get("Femme", 0)/len(df_cand)*100:.1f}%)')
print(f'  - Age : {age_valid.min()}-{age_valid.max()} ans, moyenne {age_valid.mean():.1f}, mediane {age_valid.median():.0f}')
print(f'  - Niveau d etudes principal : Bac ({edu_counts.get("NV_4_BAC", 0):,}), Bac+3 ({edu_counts.get("NV_6_BAC_3", 0):,})')
print(f'  - Nom/prenom : 100% anonymises')
print(f'  - Mobilite : 90.6% non renseignee')
print()
print(f'OFFRES :')
print(f'  - Contrats : {contrat_counts.get("CDD", 0):,} CDD ({contrat_counts.get("CDD", 0)/len(df_off)*100:.1f}%), {contrat_counts.get("CDI", 0):,} CDI ({contrat_counts.get("CDI", 0)/len(df_off)*100:.1f}%)')
print(f'  - Localisations : POINTE-NOIRE ({loc_counts.get("POINTE-NOIRE", 0):,}), BRAZZAVILLE ({loc_counts.get("BRAZZAVILLE", 0):,})')
print(f'  - Description : 94.4% manquante')
print(f'  - Competences recherchees : 94.4% manquante')
print()
print(f'DEFS POUR LE MATCHING :')
print(f'  - 94% des offres n ont pas de description ni de competences ciblees')
print(f'  - Les candidats et offres couvrent {len(common)} secteurs en commun sur {len(cand_sectors)}')
print(f'  - Mobilite geographique quasi non renseeigne chez les candidats')
print(f'  - Justifie l usage de NLP/embeddings semantiques pour le matching')
print()
print(f'SOLUTION ACPE MATCH :')
print(f'  - Pipeline NLP : normalisation -> enrichment -> embeddings (bge-m3)')
print(f'  - FAISS pour la retrieval (top-200) + CatBoost YetiRank pour le reranking')
print(f'  - 71 features de matching (skills, secteur, proximite, tension)')
print(f'  - Recherche en langage naturel (Bonus 1) + Analyse skill gap (Bonus 2)')

In [ ]:
conn.close()
print('Base fermee. EDA termine.')
print('Graphiques sauvegardes : eda_01_missing_data.png a eda_11_sector_comparison.png')